In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv()) # read local .env file

True

In [2]:
from openai import OpenAI
import json

client = OpenAI()

Model = "gpt-4.1-nano"

In [8]:
### 문제 1: 간단한 날씨 함수 호출 흐름 구성

def get_current_weather(location, unit="celsius"):
    if location == "부산" or location == 'Busan':
        return json.dumps({"location": "부산", "temperature": "24", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

weather_tools = [
    {
        "type": "function",
        "name": "get_current_weather",
        "description": "현재 날씨를 가져옵니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string"}
            },
            "required": ["location"]
        }
    }
]

user_input = [{"role": "user", "content": "부산의 날씨가 어떤지 알려줘"}]

response = client.responses.create(model=Model, input=user_input, tools=weather_tools)

tool_call = response.output[0]
args = json.loads(tool_call.arguments)
print(args)

result = get_current_weather(args['location'])

user_input.extend([
    tool_call,
    {"type": "function_call_output", "call_id": tool_call.call_id, "output": result}
])

response_2 = client.responses.create(model=Model, input=user_input, tools=weather_tools)

print("\n최종 응답:")
print(response_2.output_text)

{'location': 'Busan'}

최종 응답:
현재 부산의 날씨는 약 24도입니다. 쾌적한 하루 되세요!


In [11]:
### 문제 2: Open-Meteo API 사용
import requests

def get_weather(latitude, longitude):
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m"
    )
    return response.json()['current']['temperature_2m']

meteo_tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "현재 기온을 가져옵니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True   # JSON 스키마의 엄격한 준수(strict mode)를 적용
    }
]

response = client.responses.create(
    model=Model,
    input=[{"role": "user", "content": "서울 날씨 알려줘"}],
    tools=meteo_tools
)
tool_call = response.output[0]

args = json.loads(tool_call.arguments)
result = get_weather(args['latitude'], args['longitude'])

print("\n 현재 기온:", result)


 현재 기온: 20.9


In [12]:
### 문제 3: 이메일 함수 호출 흐름

def send_email(to, subject, body):
    return f"[이메일 전송됨] 수신자: {to}, 제목: {subject}, 내용: {body}"

email_tools = [
    {
        "type": "function",
        "name": "send_email",
        "description": "이메일을 보냅니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"]
        }
    }
]

messages = [{"role": "user", "content": "kim@example.com에게 제목은 '중요 공지', 내용은 '내일 회의가 연기되었습니다'라는 이메일을 보내줘"}]
response = client.responses.create(model=Model, input=messages, tools=email_tools)
tool_call = response.output[0]
args = json.loads(tool_call.arguments)
result = send_email(**args)

print("\n✅ 이메일 결과:", result)


✅ 이메일 결과: [이메일 전송됨] 수신자: kim@example.com, 제목: 중요 공지, 내용: 내일 회의가 연기되었습니다
